In [ ]:
# Shared project setup for imports and file locations
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
FIGURES_DIR = PROJECT_ROOT / "figures"


In [ ]:
import ast
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats

from pdm_learn.modeling import _build_predictor, _predict_scores


In [ ]:
BIOGRID_PATH = ARTIFACTS_DIR / "results" / "clean_biogrid_interactions_pdm_combined.csv"
CANCER_COMPLEXES_PATH = PROJECT_ROOT / "notebooks" / "PPI" / "Wanyi's scripts and data" / "cancer_complexes_clean.xlsx"
OUTPUT_PATH = ARTIFACTS_DIR / "results" / "ranked_PPI_pairs_cancer.csv"

METADATA_COLUMNS = {"Gene1", "Gene2", "Interaction_Type", "Confidence_Score", "pair"}
GENES_COLUMN = "Representative Genes (Core Members)"


In [ ]:
def parse_gene_list(value):
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            value = value.split(",")
    if not isinstance(value, (list, tuple, set)):
        value = [value]
    return [str(gene).strip() for gene in value if str(gene).strip()]


def canonical_pair(gene_a, gene_b):
    left, right = sorted((str(gene_a).strip(), str(gene_b).strip()))
    return f"{left}.{right}"


def cancer_complex_pair_set(cancer_complexes, genes_column=GENES_COLUMN):
    pairs = set()
    for genes in cancer_complexes[genes_column].map(parse_gene_list):
        for gene_a, gene_b in combinations(dict.fromkeys(genes), 2):
            pairs.add(canonical_pair(gene_a, gene_b))
    return pairs


def sampled_cancer_complex_pair_set(
    cancer_complexes,
    *,
    max_pairs_per_complex=5,
    available_pairs=None,
    rng=None,
    genes_column=GENES_COLUMN,
):
    rng = np.random.default_rng() if rng is None else rng
    sampled_pairs = set()

    for genes in cancer_complexes[genes_column].map(parse_gene_list):
        possible_pairs = list(combinations(dict.fromkeys(genes), 2))
        if available_pairs is not None:
            possible_pairs = [
                pair for pair in possible_pairs if canonical_pair(pair[0], pair[1]) in available_pairs
            ]
        rng.shuffle(possible_pairs)

        used_genes = set()
        selected_count = 0
        for gene_a, gene_b in possible_pairs:
            if gene_a in used_genes or gene_b in used_genes:
                continue
            sampled_pairs.add(canonical_pair(gene_a, gene_b))
            used_genes.update((gene_a, gene_b))
            selected_count += 1
            if selected_count == max_pairs_per_complex:
                break

    return sampled_pairs


def reverse_feature_source_columns(feature_columns):
    blocks = {}
    for column in feature_columns:
        prefix, index = column.rsplit(".", 1)
        blocks.setdefault(prefix, {})[int(index)] = column

    dataset_sizes = {}
    for prefix, columns in blocks.items():
        dataset_a, dataset_b = prefix.split(".", 1)
        if dataset_a == dataset_b:
            dataset_sizes[dataset_a] = int(np.sqrt(len(columns)))

    source_columns = []
    for column in feature_columns:
        prefix, index = column.rsplit(".", 1)
        dataset_a, dataset_b = prefix.split(".", 1)
        rows = dataset_sizes[dataset_a]
        columns = dataset_sizes[dataset_b]
        row, col = divmod(int(index), columns)
        source_prefix = f"{dataset_b}.{dataset_a}"
        source_columns.append(blocks[source_prefix][col * rows + row])
    return source_columns


def reverse_pair_table(data, feature_columns):
    reversed_data = data.copy()
    reversed_data[feature_columns] = data[reverse_feature_source_columns(feature_columns)].to_numpy()
    reversed_data["Gene1"] = data["Gene2"].to_numpy()
    reversed_data["Gene2"] = data["Gene1"].to_numpy()
    reversed_data["pair"] = [f"{gene_a}.{gene_b}" for gene_a, gene_b in zip(reversed_data["Gene1"], reversed_data["Gene2"])]
    return reversed_data


def choose_oriented_rows(original_values, reversed_values, row_indices, rng):
    use_reversed = rng.integers(0, 2, size=len(row_indices)).astype(bool)
    values = original_values[row_indices].copy()
    values[use_reversed] = reversed_values[row_indices][use_reversed]
    return values


def ks_feature_indices(positive_values, negative_values, features_left):
    if features_left is None:
        features_left = len(positive_values)
    features_left = max(1, min(features_left, positive_values.shape[1]))
    p_values = [stats.ks_2samp(positive_values[:, i], negative_values[:, i], method="asymp").pvalue for i in range(positive_values.shape[1])]
    return np.sort(np.argpartition(p_values, features_left - 1)[:features_left])


def rank_PPI_pairs_cancer(
    clean_biogrid_interactions_combined,
    cancer_complexes,
    *,
    trials=100,
    model="GBR",
    ks_test=True,
    features_left=None,
    max_pairs_per_complex=5,
    random_state=1,
):
    data = clean_biogrid_interactions_combined.copy()
    data["canonical_pair"] = [canonical_pair(a, b) for a, b in zip(data["Gene1"], data["Gene2"])]
    data["cancer_complex_pair"] = data["canonical_pair"].isin(cancer_complex_pair_set(cancer_complexes))

    feature_columns = [column for column in data.columns if column not in METADATA_COLUMNS | {"canonical_pair", "cancer_complex_pair"}]
    model_data = data.dropna(subset=feature_columns).reset_index(drop=True)
    reversed_data = reverse_pair_table(model_data, feature_columns)
    original_values = model_data[feature_columns].to_numpy()
    reversed_values = reversed_data[feature_columns].to_numpy()
    available_pairs = set(model_data["canonical_pair"])
    scores = np.zeros(len(model_data))
    score_iterations = np.zeros(len(model_data))
    rng = np.random.default_rng(random_state)

    for _ in range(trials):
        sampled_pairs = sampled_cancer_complex_pair_set(
            cancer_complexes,
            max_pairs_per_complex=max_pairs_per_complex,
            available_pairs=available_pairs,
            rng=rng,
        )
        positive_indices = model_data.index[model_data["canonical_pair"].isin(sampled_pairs)].to_numpy()
        candidate_indices = model_data.index[~model_data["cancer_complex_pair"]].to_numpy()
        negative_indices = rng.choice(candidate_indices, size=len(positive_indices), replace=False)
        test_indices = model_data.index.to_numpy()

        positive_values = choose_oriented_rows(original_values, reversed_values, positive_indices, rng)
        negative_values = choose_oriented_rows(original_values, reversed_values, negative_indices, rng)
        selected_features = ks_feature_indices(positive_values, negative_values, features_left) if ks_test else np.arange(len(feature_columns))

        X = np.concatenate([positive_values[:, selected_features], negative_values[:, selected_features]])
        y = np.concatenate([np.ones(len(positive_values)), np.zeros(len(negative_values))])
        predictor = _build_predictor(model).fit(X, y)

        forward_scores = _predict_scores(predictor, original_values[test_indices][:, selected_features])
        reverse_scores = _predict_scores(predictor, reversed_values[test_indices][:, selected_features])
        scores[test_indices] += np.maximum(forward_scores, reverse_scores)
        score_iterations[test_indices] += 1

    average_scores = np.divide(scores, score_iterations, out=np.full_like(scores, np.nan), where=score_iterations > 0)
    output_columns = ["Gene1", "Gene2", "pair", "canonical_pair", "cancer_complex_pair"]
    output = model_data[output_columns].assign(score=average_scores, score_iterations=score_iterations.astype(int))
    return output.sort_values("score", ascending=False).reset_index(drop=True)


In [ ]:
clean_biogrid_interactions_combined = pd.read_csv(BIOGRID_PATH, low_memory=False)
cancer_complexes = pd.read_excel(CANCER_COMPLEXES_PATH, index_col=0)

positive_pairs = cancer_complex_pair_set(cancer_complexes)
matched_pairs = {
    canonical_pair(gene_a, gene_b)
    for gene_a, gene_b in zip(clean_biogrid_interactions_combined["Gene1"], clean_biogrid_interactions_combined["Gene2"])
} & positive_pairs

pd.Series(
    {
        "biogrid_pairs": len(clean_biogrid_interactions_combined),
        "cancer_complex_pairs": len(positive_pairs),
        "cancer_complex_pairs_in_biogrid": len(matched_pairs),
    }
)


biogrid_pairs                      85108
cancer_complex_pairs                1064
cancer_complex_pairs_in_biogrid     1007
dtype: int64

In [ ]:
ranking = rank_PPI_pairs_cancer(
    clean_biogrid_interactions_combined,
    cancer_complexes,
    trials=1000,
    model="GBR",
    ks_test=True,
)
ranking.head(20)


,Gene1,Gene2,pair,canonical_pair,cancer_complex_pair,score,score_iterations
0,EZH2,SMARCA4,EZH2.SMARCA4,EZH2.SMARCA4,False,1.007106,10000
1,EZH2,SUZ12,EZH2.SUZ12,EZH2.SUZ12,True,1.004931,10000
2,PLK1,RRM2,PLK1.RRM2,PLK1.RRM2,False,1.001022,10000
3,KMT2A,DOT1L,KMT2A.DOT1L,DOT1L.KMT2A,False,0.996670,10000
4,MAPK1,CREBBP,MAPK1.CREBBP,CREBBP.MAPK1,False,0.992587,10000
5,ARID1B,SMARCA4,ARID1B.SMARCA4,ARID1B.SMARCA4,True,0.991566,10000
6,CTNNB1,CREBBP,CTNNB1.CREBBP,CREBBP.CTNNB1,False,0.991470,10000
7,KRAS,MAPK1,KRAS.MAPK1,KRAS.MAPK1,True,0.990165,10000
8,SMARCA4,ARID1A,SMARCA4.ARID1A,ARID1A.SMARCA4,True,0.989872,10000
9,CTNNB1,KMT2D,CTNNB1.KMT2D,CTNNB1.KMT2D,False,0.988426,10000


In [ ]:
ranking

,Gene1,Gene2,pair,canonical_pair,cancer_complex_pair,score,score_iterations
0,EZH2,SMARCA4,EZH2.SMARCA4,EZH2.SMARCA4,False,1.007106,10000
1,EZH2,SUZ12,EZH2.SUZ12,EZH2.SUZ12,True,1.004931,10000
2,PLK1,RRM2,PLK1.RRM2,PLK1.RRM2,False,1.001022,10000
3,KMT2A,DOT1L,KMT2A.DOT1L,DOT1L.KMT2A,False,0.996670,10000
4,MAPK1,CREBBP,MAPK1.CREBBP,CREBBP.MAPK1,False,0.992587,10000
...,...,...,...,...,...,...,...
85076,DNAJC16,FKBP8,DNAJC16.FKBP8,DNAJC16.FKBP8,False,0.202373,10000
85077,NDUFA7,MRRF,NDUFA7.MRRF,MRRF.NDUFA7,False,0.198116,10000
85078,FBXL3,TCAP,FBXL3.TCAP,FBXL3.TCAP,False,0.196630,10000
85079,HIST1H1T,HIST1H2AB,HIST1H1T.HIST1H2AB,HIST1H1T.HIST1H2AB,False,0.192409,10000


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
ranking.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH


PosixPath('/Users/jzx/Documents/Computer Science/PDM_Learn/artifacts/results/ranked_PPI_pairs_cancer.csv')